<a href="https://colab.research.google.com/github/LucasDiiorio/Problem-Set-1-Econ_5200_PS1.ipynb/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [ ]:
# Clean consumer transactions
n = 20_000
consumers = pd.DataFrame({
    "customer_id": rng.choice(np.arange(1, 5001), size=n),
    "year": rng.integers(1, 3, size=n),
    "basket_value": rng.lognormal(mean=np.log(40), sigma=0.6, size=n),
})
consumers["segment"] = "consumer"
print(np.median(consumers["basket_value"]))

40.17592745393486


In [ ]:
# The truth: the consumer average per year, recorded before contamination
true_avg = consumers.groupby("year")["basket_value"].mean()
true_avg

,basket_value
year,
1,48.279304
2,48.296859


In [ ]:
# Contamination 1: B2B orders
n_b2b = 600
b2b = pd.DataFrame({
    "customer_id": rng.choice(np.arange(9001, 9101), size=n_b2b),
    "year": rng.choice([1, 2], size=n_b2b, p=[0.3, 0.7]),
    "basket_value": rng.lognormal(mean=np.log(800), sigma=0.7, size=n_b2b),
})
b2b["segment"] = "b2b"
data = pd.concat([consumers, b2b], ignore_index=True)

In [ ]:
# Contamination 2: what gets logged changes between year 1 and year 2
not_logged = (data["year"] == 2) & (data["basket_value"] < 15)
logged = data[~not_logged].copy()

print(logged.groupby("year")["basket_value"].mean())
print(true_avg)

year
1    66.607111
2    90.524600
Name: basket_value, dtype: float64
year
1    48.279304
2    48.296859
Name: basket_value, dtype: float64


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [ ]:
# The dashboard's number vs the truth
dashboard_avg = logged.groupby("year")["basket_value"].mean()

comparison = pd.DataFrame({
    "dashboard": dashboard_avg,
    "truth": true_avg,
})
comparison["gap_$"] = comparison["dashboard"] - comparison["truth"]
comparison["gap_%"] = 100 * comparison["gap_$"] / comparison["truth"]
print(comparison.round(2))

      dashboard  truth  gap_$  gap_%
year                                
1         66.61  48.28  18.33  37.96
2         90.52  48.30  42.23  87.43


In [ ]:
dash_yoy = 100 * (dashboard_avg[2] / dashboard_avg[1] - 1)
true_yoy = 100 * (true_avg[2] / true_avg[1] - 1)
print(f"\nDashboard YoY change: {dash_yoy:+.1f}%")
print(f"True YoY change:      {true_yoy:+.1f}%")


Dashboard YoY change: +35.9%
True YoY change:      +0.0%


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [ ]:
# How much of the gap is the skew and the B2B tail?
mean_truth = consumers.groupby("year")["basket_value"].mean()
mean_b2b   = data.groupby("year")["basket_value"].mean()

b2b_contrib = mean_b2b - mean_truth
print("B2B tail contribution ($):")
print(b2b_contrib.round(2))

B2B tail contribution ($):
year
1    18.33
2    38.33
Name: basket_value, dtype: float64


In [ ]:
# How much is the change in what was logged?
mean_logged = logged.groupby("year")["basket_value"].mean()
logging_contrib = mean_logged - mean_b2b
print("Logging change contribution ($):")
print(logging_contrib.round(2))

Logging change contribution ($):
year
1    0.0
2    3.9
Name: basket_value, dtype: float64


In [ ]:
total_gap = mean_logged - mean_truth
summary = pd.DataFrame({
    "b2b_tail": b2b_contrib,
    "logging": logging_contrib,
    "total_gap": total_gap,
    "share_b2b_%": 100 * b2b_contrib / total_gap,
    "share_logging_%": 100 * logging_contrib / total_gap,
})
print(summary.round(2))

      b2b_tail  logging  total_gap  share_b2b_%  share_logging_%
year                                                            
1        18.33      0.0      18.33       100.00             0.00
2        38.33      3.9      42.23        90.76             9.24


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |Y1=+$18.33,Y2= +$38.33 |Compared cleaned consumer mean to the mean after addinng B2B but before the log changes |
| Change in what was logged |Y1=$0.00,Y2=$+3.90 |Compared the mean before and after applying the year-2 logging filter, with B2B orders already included. |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [ ]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
cust_median = logged.groupby("customer_id")["basket_value"].median()
b2b_ids = cust_median[cust_median > 300].index
excluded = logged[~logged["customer_id"].isin(b2b_ids)]
g = logged.groupby("year")["basket_value"]

estimators = pd.DataFrame({
    "truth": true_avg,
    "median": g.median(),
    "trimmed mean": g.apply(lambda x: stats.trim_mean(x, 0.1)),
    "mean after B2B excl.": excluded.groupby("year")["basket_value"].mean(),
})
print(estimators.round(2))

      truth  median  trimmed mean  mean after B2B excl.
year                                                   
1     48.28   40.54         44.41                 48.28
2     48.30   43.22         47.87                 50.28


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median |Y1=40.54,Y2=43.22 | Utilizes the typical basket not the avg|Highly skewed data |
| Trimmed mean |Y1=44.41,Y2=47.87 |Extreme values on both ends are errors and roughly symetric |Highly skewed, extreme values are features not errors |
| Mean after B2B exclusion |Y1=48.28,Y2=50.28 |Assumes we correctly classified consumers and every basket was logged | We misclassifiy consumers or we have unlogged small consumer baskets inflating the data|

**Which number should the dashboard show?** Commit to one and defend it.

*The dashboard should show the mean after B2B exclusion. This mean is suppose to represent the average consumer basket.The median and trimmed mean measures a right skewed baske of data. This means that their means fall below the true average even with no contamination. Meanwhile the mean after B2B exclusion is $48.28 in year which is identical to identical to the truth *

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/basket_metrics.py
"""Outlier Detection: Transaction Data. Audit columns and estimate a robust average basket."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame, iqr_k: float = 1.5) -> pd.DataFrame:
    """Audit the columns of a DataFrame"""
    rows = []
    for col in df.columns:
        s = df[col]
        row = {
            "column": col,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "pct_missing": round(100 * s.isna().mean(), 2),
            "skew": np.nan,
            "n_outliers": np.nan,
        }
        if pd.api.types.is_numeric_dtype(s):
            clean = s.dropna()
            q1, q3 = clean.quantile([0.25, 0.75])
            iqr = q3 - q1
            outside = (clean < q1 - iqr_k * iqr) | (clean > q3 + iqr_k * iqr)
            row["skew"] = round(float(clean.skew()), 3)
            row["n_outliers"] = int(outside.sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("column")


def robust_mean(
    x,
    method: str = "median",
    trim: float = 0.1,
    customer_ids=None,
    b2b_threshold: float = 300.0,
) -> float:

    values = pd.Series(np.asarray(x, dtype=float))

    if method == "median":
        return float(values.median())

    if method == "trimmed":
        return float(stats.trim_mean(values.dropna(), trim))

    if method == "exclusion":
        if customer_ids is None:
            return float(values[values <= b2b_threshold].mean())
        ids = pd.Series(np.asarray(customer_ids))
        if len(ids) != len(values):
            raise ValueError("customer_ids must be the same length as x.")
        cust_median = values.groupby(ids).median()
        b2b = cust_median[cust_median > b2b_threshold].index
        return float(values[~ids.isin(b2b)].mean())

    raise ValueError(
        f"Unknown method {method!r}; use 'median', 'trimmed' or 'exclusion'."
    )


if __name__ == "__main__":
    # A short demonstration that runs when the file is run as a script
    rng = np.random.default_rng(0)
    consumers = pd.DataFrame({
        "customer_id": rng.integers(1, 1001, size=5000),
        "basket_value": rng.lognormal(np.log(40), 0.6, size=5000),
    })
    b2b = pd.DataFrame({
        "customer_id": rng.integers(9001, 9021, size=150),
        "basket_value": rng.lognormal(np.log(800), 0.7, size=150),
    })
    demo = pd.concat([consumers, b2b], ignore_index=True)

    print("Audit report:")
    print(audit_report(demo))
    print(f"\nTrue consumer mean: {consumers['basket_value'].mean():.2f}")
    print(f"Naive mean:         {demo['basket_value'].mean():.2f}")
    for m in ["median", "trimmed", "exclusion"]:
        est = robust_mean(demo["basket_value"], method=m,
                          customer_ids=demo["customer_id"])
        print(f"{m:<19} {est:.2f}")

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [ ]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance

<module 'basket_metrics' from '/content/src/basket_metrics.py'>

---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**
2. "Why 5%? You picked the trim that gives the answer you wanted."
Type: the statistic.
Any analytically minded board member will ask this, and it's harder to defend than it sounds. The issue is redundancy. If the B2B rule correctly removes the skewed tail, the trim has little left to do, so why include it? If the trim is still doing a lot of work after the exclusion, then either the B2B rule misses part of the tail, or the trim is cutting genuine high-value consumer baskets. Those baskets are exactly what a premiumisation strategy would want to see. A symmetric trim also drops the smallest baskets, which shifts the level for reasons unrelated to B2B. There is also a practical problem. Trimmed means don't aggregate. Store-level trimmed means won't roll up to the regional figure, and the dashboard will show inconsistent totals the first time someone drills down.
The evidence that settles it is a sensitivity table. After the B2B exclusion, show the level and the year-over-year change at 0%, 1%, 2.5%, 5%, and 10% trims, alongside the median. If the YoY direction and rough size hold across the range, the trim choice doesn't matter, and you can say so. If the story changes with the trim level, the board needs to know, and you probably want the median or an untrimmed post-exclusion mean instead. Separately, profile the baskets the trim removes after the exclusion. If they look like unflagged B2B (large quantities, repeat accounts, trade SKUs), fix the B2B rule rather than trimming.



**What you found, including if it changes your Phase 2 recommendation:**

*Your answer here.*

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

*Your slide here.*

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors